In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error

tr = pd.read_csv("../data/processed/split_train.csv")
te = pd.read_csv("../data/processed/split_test.csv")
for d in (tr, te):
    d["text"] = d["text"].fillna("")
    d["sub_category"] = d["sub_category"].fillna("unknown")

features = ["text", "main_category", "sub_category"]
y_true = np.expm1(te["log_price"])


In [2]:
pre = ColumnTransformer([
    ("text", TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=3), "text"),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["main_category", "sub_category"]),
])

rf = Pipeline([
    ("pre", pre),
    ("reg", RandomForestRegressor(
        n_estimators=100, min_samples_leaf=3,
        max_features="sqrt", n_jobs=2, random_state=42)),
])

rf.fit(tr[features], tr["log_price"])

pred = rf.predict(te[features])
p = np.expm1(pred)
print("R²:", round(r2_score(te["log_price"], pred), 3))
print(f"mean error ${mean_absolute_error(y_true, p):.2f} | median error ${median_absolute_error(y_true, p):.2f}")

R²: 0.413
mean error $43.07 | median error $7.49


In [3]:
names = rf.named_steps["pre"].get_feature_names_out()
imp = rf.named_steps["reg"].feature_importances_
top = np.argsort(imp)[-15:][::-1]
for i in top:
    print(f"{names[i]:40s} {imp[i]:.4f}")

cat__sub_category_Furniture              0.0196
cat__sub_category_Outdoor Recreation     0.0145
cat__main_category_Sports & Outdoors     0.0127
cat__sub_category_Arts & Crafts          0.0121
cat__sub_category_Party Supplies         0.0097
cat__sub_category_Tricycles, Scooters & Wagons 0.0096
text__party                              0.0096
cat__main_category_Toys & Games          0.0083
text__slide                              0.0083
cat__main_category_Home & Kitchen        0.0066
text__4wd                                0.0061
text__table                              0.0061
text__with                               0.0060
text__skate                              0.0059
text__01                                 0.0057
